In [1]:
import os

In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

## Understanding Dot Product and Cosine Similarity between Word Embeddings

### **1. Introduction**

In modern Natural Language Processing (NLP), words, phrases, or sentences are often represented as **vectors** — called **embeddings**.
These embeddings capture **semantic meaning**, so that words with similar meanings lie close to each other in a high-dimensional space.

For example, words like *“King”* and *“Queen”* should be closer in vector space than *“King”* and *“Table.”*

---

### **2. The Role of the Dot Product**

The **dot product** measures how much two vectors point in the **same direction**.
For vectors **A** and **B**, it’s defined as:

$$
A \cdot B = |A|, |B| \cos(\theta)
$$

* ( |A| ) and ( |B| ) are the **magnitudes (lengths)** of the vectors.
* ( \theta ) is the **angle** between them.

If the dot product is **large and positive**, the vectors point in **similar directions**, meaning the words are **semantically related**.
If it’s **near zero**, the words are **unrelated**.
If it’s **negative**, they are **oppositely related** (rare in word embeddings).

---

### **3. Cosine Similarity**

The **cosine similarity** normalizes the dot product to remove the effect of vector length.
It focuses **only on the angle** between vectors and is defined as:

$$
\text{Cosine Similarity} = \frac{A \cdot B}{|A|,|B|}
$$

* **Range:** −1 to 1

  * `1` → perfectly aligned (meanings are nearly identical)
  * `0` → unrelated
  * `−1` → completely opposite

In word embeddings:

* **High cosine similarity (close to 1)** → words have **similar meanings**.
* **Low cosine similarity (close to 0)** → words are **semantically distant**.

---

### **4. Example: “King” and “Queen”**

When you encode:

```python
question = "King"
document = "Queen"
```

and calculate their **cosine similarity**, the resulting score (e.g., `0.85`) means:

* The two vectors are **strongly aligned**, forming a **small angle** between them.
* Hence, the model understands that *“King”* and *“Queen”* are **semantically related** — both represent **royalty** but differ in gender.

If you convert the cosine similarity to an **angle**, smaller angles imply greater semantic closeness:

$$
\theta = \cos^{-1}(0.85) \approx 31.8^\circ
$$

---

### **5. Why It Matters**

Understanding dot product and cosine similarity helps you:

* **Quantify semantic similarity** between words or sentences.
* **Compare relationships** (e.g., *“King–Queen” ≈ “Man–Woman”*).
* **Build systems** like semantic search, question answering, or RAG models that rely on **contextual similarity**.

---

### **6. Summary**

| Concept               | Purpose                                    | Output Type      | Interpretation                       |
| --------------------- | ------------------------------------------ | ---------------- | ------------------------------------ |
| **Dot Product**       | Measures directional alignment + magnitude | Scalar           | Larger value = more aligned          |
| **Cosine Similarity** | Measures pure directional alignment        | Scalar (−1 to 1) | Higher = more semantically similar   |
| **Application**       | NLP embeddings                             | —                | Reveals how related two meanings are |

---

So, the topic **“Understanding Dot Product and Cosine Similarity between Word Embeddings”** means exploring how mathematical relationships between vectors — through dot product and cosine similarity — help models detect **semantic relationships** between words in embedding space.

In [4]:
# Encode both the question and document into dense vector representations
question = "King"
document = "Queen"

# Convert text into numerical embeddings (high-dimensional vectors)
question_vect = model.encode(question)
docs_vect = model.encode(document)

# Compute the dot product between the two embeddings
# This measures how semantically similar the two sentences are
similarity_score = np.dot(question_vect, docs_vect)

# Display results
print("=== Embedding Similarity Analysis ===")
print(f"Question: {question}")
print(f"Document: {document}")
print(f"\nQuestion Vector (first 5 values): {question_vect[:5]}")
print(f"Document Vector (first 5 values): {docs_vect[:5]}")

print("\nFormula: Similarity = |v1| × |v2| × cos(θ)")
print(f"Dot Product (Similarity Score): {similarity_score:.6f}")

# Optional: If you want to show the cosine similarity explicitly
norm_question = np.linalg.norm(question_vect)
norm_docs = np.linalg.norm(docs_vect)
cosine_similarity = similarity_score / (norm_question * norm_docs)
angle_radians = np.arccos(cosine_similarity)
angle_degrees = np.degrees(angle_radians)
print(f"Cosine Similarity (Normalized): {cosine_similarity:.6f}")
print("\nConclusion:")
print(f"This means the 'question vector' and 'document vector' are nearly aligned, forming an angle of approximately 0.589328 radians (≈ {angle_degrees:.2f}°) between them — indicating that they are semantically close in meaning.")

=== Embedding Similarity Analysis ===
Question: King
Document: Queen

Question Vector (first 5 values): [-0.05959932  0.05051239 -0.06951008  0.07968022 -0.0467477 ]
Document Vector (first 5 values): [ 0.03548699 -0.06560468 -0.00993496  0.03159032 -0.01338684]

Formula: Similarity = |v1| × |v2| × cos(θ)
Dot Product (Similarity Score): 0.680713
Cosine Similarity (Normalized): 0.680713

Conclusion:
This means the 'question vector' and 'document vector' are nearly aligned, forming an angle of approximately 0.589328 radians (≈ 47.10°) between them — indicating that they are semantically close in meaning.


## Measuring Semantic and Relational Similarity using Embedding Vectors
1. **`calculate_similarity_score`**

   * Computes how semantically close two words or phrases are.
   * Uses cosine similarity as the metric.

2. **Standard deviation among scores**

   * Checks if the model treats all gender/relational pairs similarly.
   * Smaller values → more consistency.

3. **Difference vectors (`get_difference_vect`)**

   * Represent relationships rather than meanings.
   * Example: “King − Queen” vector captures a “male-to-female royalty” direction.

4. **Pairwise comparison**

   * Measures whether all such relationships (King–Queen, Male–Female, Men–Women) point in a similar direction.

5. **Conclusion section**

   * Summarizes what the results imply about your embedding model’s understanding of relationships.

In [5]:
import numpy as np

# Function to calculate cosine similarity between two text inputs
def calculate_similarity_score(model, question, docs):
    # Encode both inputs into embedding vectors
    question_vect = model.encode(question)
    docs_vect = model.encode(docs)
    
    # Calculate their magnitudes (vector lengths)
    question_vect_magnitude = np.linalg.norm(question_vect)
    docs_vect_magnitude = np.linalg.norm(docs_vect)

    # Compute the dot product and then derive cosine similarity
    dot_product = np.dot(question_vect, docs_vect)
    cosine_similarity = dot_product / (question_vect_magnitude * docs_vect_magnitude)

    return cosine_similarity


# Comparing how semantically close certain word pairs are
men_women_similarity_score = calculate_similarity_score(model, "Men", "Women")
king_queen_similarity_score = calculate_similarity_score(model, "King", "Queen")
male_female_similarity_score = calculate_similarity_score(model, "Male", "Female")

# Store and analyze the variation between these similarity scores
similarity_scores = [men_women_similarity_score, king_queen_similarity_score, male_female_similarity_score]
similarity_std = np.std(similarity_scores)

print("=== Semantic Similarity Results ===")
print(f"Men ↔ Women Similarity: {men_women_similarity_score:.6f}")
print(f"King ↔ Queen Similarity: {king_queen_similarity_score:.6f}")
print(f"Male ↔ Female Similarity: {male_female_similarity_score:.6f}")
print(f"Standard Deviation between these similarities: {similarity_std:.6f}")
print("Lower standard deviation indicates that the model captures gender or relational patterns consistently.\n")


# Function to find the difference vector between two related words
def get_difference_vect(model, word1, word2):
    word1_vect = model.encode(word1)
    word2_vect = model.encode(word2)
    return word1_vect - word2_vect


# Function to calculate cosine similarity between two difference vectors
def calculate_cosine_similarity_score_vect(v1, v2):
    dot_product = np.dot(v1, v2)
    v1_magnitude = np.linalg.norm(v1)
    v2_magnitude = np.linalg.norm(v2)
    cosine_similarity = dot_product / (v1_magnitude * v2_magnitude)
    return cosine_similarity


# Generate difference vectors (relationship direction in vector space)
kq = get_difference_vect(model, "King", "Queen")
mf = get_difference_vect(model, "Male", "Female")
mw = get_difference_vect(model, "Men", "Women")

# Store all relationship vectors
vectors = [kq, mf, mw]
vector_names = ["King–Queen", "Male–Female", "Men–Women"]

# Compute pairwise cosine similarities between relationship vectors
print("=== Relationship Vector Similarity Matrix ===")
cosine_similarities = []
for i, v1 in enumerate(vectors):
    for j, v2 in enumerate(vectors):
        score = calculate_cosine_similarity_score_vect(v1, v2)
        cosine_similarities.append(score)
        print(f"{vector_names[i]} ↔ {vector_names[j]}: {score:.6f}")

# Analyze the variation across all pairwise similarity scores
relationship_std = np.std(np.array(cosine_similarities))
print(f"\nStandard Deviation of all relationship similarities: {relationship_std:.6f}")

print("\nConclusion:")
print("If the pairwise similarities are high and the standard deviation is low, "
      "it suggests that the model captures consistent relational patterns between these word pairs. "
      "In this context, the vectors representing 'King–Queen', 'Male–Female', and 'Men–Women' "
      "point in nearly the same semantic direction, showing that the model understands "
      "the analogy or gender-based relationship between them.")

=== Semantic Similarity Results ===
Men ↔ Women Similarity: 0.719090
King ↔ Queen Similarity: 0.680713
Male ↔ Female Similarity: 0.733793
Standard Deviation between these similarities: 0.022377
Lower standard deviation indicates that the model captures gender or relational patterns consistently.

=== Relationship Vector Similarity Matrix ===
King–Queen ↔ King–Queen: 1.000000
King–Queen ↔ Male–Female: 0.378769
King–Queen ↔ Men–Women: 0.331722
Male–Female ↔ King–Queen: 0.378769
Male–Female ↔ Male–Female: 1.000000
Male–Female ↔ Men–Women: 0.790963
Men–Women ↔ King–Queen: 0.331722
Men–Women ↔ Male–Female: 0.790963
Men–Women ↔ Men–Women: 1.000000

Standard Deviation of all relationship similarities: 0.289516

Conclusion:
If the pairwise similarities are high and the standard deviation is low, it suggests that the model captures consistent relational patterns between these word pairs. In this context, the vectors representing 'King–Queen', 'Male–Female', and 'Men–Women' point in nearly the s